# 论文后处理流程：矢量化 + 3D 重建
## Automatic Reconstruction of Semantic 3D Models from 2D Floor Plans (MVA 2023)

### 对应论文章节
```
Section 2.4 Wall Extraction（墙体矢量化）
  Step 1: 形态学预处理（去噪、填洞）
  Step 2: Canny + Hough 变换找墙体角度
  Step 3: 旋转 mask，形态学分离水平/垂直线段
  Step 4: 轮廓提取得到单个墙体段
  Step 5: Shrinking 算法（不规则 → 矩形 bbox）
  Step 6: 重叠处理（完全包含的删除，部分重叠的裁剪）

Section 2.5 3D Reconstruction（3D 重建）
  Step 1: 门窗匹配到对应墙体段
  Step 2: 用矩形 box primitives 构建墙体 3D 模型
  Step 3: 在墙体上裁剪门窗开口
  Step 4: 导出 GLB/IFC
```

## 0. 环境配置

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, sys, json, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

# 3D 重建依赖
try:
    import trimesh
    HAS_TRIMESH = True
except ImportError:
    HAS_TRIMESH = False
    print('trimesh 未安装，3D 重建不可用')
    print('安装: pip install trimesh')

# ── 路径配置 ──
if os.path.exists('/workspace/production_3d'):
    CHECKPOINT_DIR = '/workspace/production_3d/checkpoints_paper'
    OUTPUT_DIR     = '/workspace/production_3d/outputs'
    CUBICASA_ROOT  = '/workspace/CubiCasa5k'
    DATA_FOLDER    = '/workspace/data/cubicasa5k/'
elif os.path.exists('/content'):
    CHECKPOINT_DIR = '/content/checkpoints_paper'
    OUTPUT_DIR     = '/content/outputs'
    CUBICASA_ROOT  = '/content/CubiCasa5k'
    DATA_FOLDER    = '/content/data/cubicasa5k/'
else:
    CHECKPOINT_DIR = './checkpoints_paper'
    OUTPUT_DIR     = './outputs'
    CUBICASA_ROOT  = r'E:\JOB\CubiCasa5k'
    DATA_FOLDER    = r'C:/Users/kawayi_yaling/.cache/kagglehub/datasets/qmarva/cubicasa5k/versions/4/cubicasa5k/cubicasa5k/'

os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('✓ 环境配置完成')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'  OUTPUT_DIR    : {OUTPUT_DIR}')
print(f'  trimesh       : {HAS_TRIMESH}')

## 1. 加载训练好的模型

In [ ]:
sys.path.insert(0, CUBICASA_ROOT)
os.chdir(CUBICASA_ROOT)

from floortrans.loaders.house import House

# 重新定义模型（和训练时一致）
from collections import OrderedDict
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import FeaturePyramidNetwork
from torchvision.models.detection.rpn import (
    AnchorGenerator, RPNHead, RegionProposalNetwork
)
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.image_list import ImageList
from torchvision import transforms


class SharedBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        resnet = resnet50(weights=None)
        self.layer0 = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        )
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x0 = self.layer0(x)
        c1 = self.layer1(x0)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return OrderedDict([('0', c1), ('1', c2), ('2', c3), ('3', c4)])


class SegmentationHead(nn.Module):
    def __init__(self, fpn_out_channels=256, num_classes=2):
        super().__init__()
        self.fpn = FeaturePyramidNetwork(
            in_channels_list=[256, 512, 1024, 2048],
            out_channels=fpn_out_channels,
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(fpn_out_channels, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, 1),
        )

    def forward(self, features, input_size):
        fpn_out = self.fpn(features)
        target_size = fpn_out['0'].shape[-2:]
        fused = fpn_out['0']
        for k in ['1', '2', '3']:
            fused = fused + F.interpolate(
                fpn_out[k], size=target_size,
                mode='bilinear', align_corners=False
            )
        logits = self.decoder(fused)
        return F.interpolate(
            logits, size=input_size,
            mode='bilinear', align_corners=False
        )


class FloorplanModel(nn.Module):
    def __init__(self, fpn_out_channels=256, seg_num_classes=2, det_num_classes=3):
        super().__init__()
        self.backbone = SharedBackbone()
        self.seg_head  = SegmentationHead(fpn_out_channels, seg_num_classes)
        self.det_fpn   = FeaturePyramidNetwork(
            in_channels_list=[256, 512, 1024, 2048],
            out_channels=fpn_out_channels,
        )
        rpn_anchor = AnchorGenerator(
            sizes=((16,), (32,), (64,), (128,)),
            aspect_ratios=((0.5, 1.0, 2.0),) * 4,
        )
        rpn_head = RPNHead(
            fpn_out_channels,
            rpn_anchor.num_anchors_per_location()[0]
        )
        self.rpn = RegionProposalNetwork(
            rpn_anchor, rpn_head,
            fg_iou_thresh=0.7, bg_iou_thresh=0.3,
            batch_size_per_image=256, positive_fraction=0.5,
            pre_nms_top_n={'training': 2000, 'testing': 1000},
            post_nms_top_n={'training': 2000, 'testing': 300},
            nms_thresh=0.7,
        )
        box_roi_pool = MultiScaleRoIAlign(
            featmap_names=['0', '1', '2', '3'],
            output_size=7, sampling_ratio=2,
        )
        box_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fpn_out_channels * 7 * 7, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, 1024), nn.ReLU(inplace=True),
        )
        box_predictor = FastRCNNPredictor(1024, det_num_classes)
        self.roi_heads = RoIHeads(
            box_roi_pool, box_head, box_predictor,
            fg_iou_thresh=0.5, bg_iou_thresh=0.5,
            batch_size_per_image=512, positive_fraction=0.25,
            bbox_reg_weights=None,
            score_thresh=0.5, nms_thresh=0.5,
            detections_per_img=100,
        )
        self.ImageList = ImageList

    def forward(self, images):
        input_size = images.shape[-2:]
        features   = self.backbone(images)
        seg_logits = self.seg_head(features, input_size)
        det_feat   = self.det_fpn(features)
        img_list   = self.ImageList(images, [images.shape[-2:]] * images.shape[0])
        proposals, _ = self.rpn(img_list, det_feat, None)
        det_out, _   = self.roi_heads(det_feat, proposals, [images.shape[-2:]] * images.shape[0], None)
        return {'seg_logits': seg_logits, 'det_outputs': det_out}


# 加载权重
model = FloorplanModel().to(DEVICE)
ckpt  = torch.load(f'{CHECKPOINT_DIR}/best_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'✓ 模型加载完成')
print(f'  epoch       : {ckpt["epoch"]}')
print(f'  val_wall_iou: {ckpt["val_wall_iou"]:.4f}')
print(f'  det_f1      : {ckpt["det_f1"]:.4f}')

# 归一化参数
NORM_MEAN = (0.485, 0.456, 0.406)
NORM_STD  = (0.229, 0.224, 0.225)
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

## 2. 推理函数

In [ ]:
def predict(image: np.ndarray,
            tile_size: int = 512,
            tile_overlap: int = 64,
            det_score_thresh: float = 0.5) -> dict:
    """
    对单张平面图做完整推理
    使用滑动窗口处理大图（Section 2.1）
    返回完整尺寸的 wall mask 和门窗 bbox
    """
    h, w = image.shape[:2]
    wall_prob = np.zeros((h, w), dtype=np.float32)
    wall_cnt  = np.zeros((h, w), dtype=np.float32)
    all_boxes, all_scores, all_labels = [], [], []

    stride = tile_size - tile_overlap
    ys = list(range(0, max(h - tile_size + 1, 1), stride))
    xs = list(range(0, max(w - tile_size + 1, 1), stride))
    if not ys or ys[-1] + tile_size < h: ys.append(max(h - tile_size, 0))
    if not xs or xs[-1] + tile_size < w: xs.append(max(w - tile_size, 0))

    for ty in ys:
        for tx in xs:
            tile = image[ty:ty+tile_size, tx:tx+tile_size].copy()
            th, tw = tile.shape[:2]

            # Padding
            if th < tile_size or tw < tile_size:
                tile = cv2.copyMakeBorder(
                    tile, 0, tile_size-th, 0, tile_size-tw,
                    cv2.BORDER_REFLECT_101
                )
            tile = cv2.resize(tile, (tile_size, tile_size),
                              interpolation=cv2.INTER_LINEAR)

            tensor = img_transform(tile).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                out = model(tensor)

            # 分割结果（wall 概率图）
            prob = torch.softmax(out['seg_logits'], dim=1)[0, 1].cpu().numpy()
            prob_resized = cv2.resize(prob, (tw, th),
                                      interpolation=cv2.INTER_LINEAR)
            wall_prob[ty:ty+th, tx:tx+tw] += prob_resized[:th, :tw]
            wall_cnt[ty:ty+th,  tx:tx+tw] += 1

            # 检测结果（门窗 bbox）
            det = out['det_outputs'][0]
            if len(det['boxes']) > 0:
                boxes  = det['boxes'].cpu().numpy()
                scores = det['scores'].cpu().numpy()
                labels = det['labels'].cpu().numpy()
                keep   = scores >= det_score_thresh
                # 坐标换算回原图
                scale_x = tw / tile_size
                scale_y = th / tile_size
                for box, score, label in zip(boxes[keep], scores[keep], labels[keep]):
                    x1 = box[0] * scale_x + tx
                    y1 = box[1] * scale_y + ty
                    x2 = box[2] * scale_x + tx
                    y2 = box[3] * scale_y + ty
                    all_boxes.append([x1, y1, x2, y2])
                    all_scores.append(score)
                    all_labels.append(label)

    # 平均重叠区域
    wall_cnt   = np.maximum(wall_cnt, 1)
    wall_prob /= wall_cnt
    wall_mask  = (wall_prob > 0.5).astype(np.uint8)

    # NMS 合并检测框
    if all_boxes:
        from torchvision.ops import nms
        boxes_t  = torch.tensor(all_boxes,  dtype=torch.float32)
        scores_t = torch.tensor(all_scores, dtype=torch.float32)
        labels_t = torch.tensor(all_labels, dtype=torch.int64)
        keep     = nms(boxes_t, scores_t, iou_threshold=0.5)
        all_boxes  = boxes_t[keep].numpy()
        all_scores = scores_t[keep].numpy()
        all_labels = labels_t[keep].numpy()
    else:
        all_boxes  = np.zeros((0, 4))
        all_scores = np.zeros(0)
        all_labels = np.zeros(0, dtype=np.int64)

    return {
        'wall_mask':  wall_mask,
        'wall_prob':  wall_prob,
        'boxes':      all_boxes,
        'scores':     all_scores,
        'labels':     all_labels,  # 1=door, 2=window
    }


print('✓ 推理函数定义完成')

## 3. Section 2.4 Step 1：形态学预处理

In [ ]:
def morphological_preprocessing(wall_mask: np.ndarray) -> np.ndarray:
    """
    论文 Section 2.4 第一步：形态学预处理
    1. morphological opening  → 去除小噪点
    2. Gaussian blur          → 平滑
    3. morphological closing  → 填充墙体内部小洞
    """
    mask = wall_mask.astype(np.uint8)

    # Step 1: Opening（去噪）
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    # Step 2: Gaussian blur（平滑边界）
    mask_f = cv2.GaussianBlur(
        mask.astype(np.float32), (5, 5), sigmaX=1.0
    )
    mask = (mask_f > 0.5).astype(np.uint8)

    # Step 3: Closing（填洞）
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    return mask


print('✓ 形态学预处理函数定义完成')

## 4. Section 2.4 Step 2：Canny + Hough 找墙体角度

In [ ]:
def find_wall_angles(wall_mask: np.ndarray,
                     angle_resolution: float = 1.0,
                     n_angles: int = 4) -> List[float]:
    """
    论文 Section 2.4：
    用 Canny 边缘检测 + Hough 变换找墙体主要方向
    大多数墙体是水平或垂直的，但也要考虑斜墙
    返回角度列表（度）
    """
    # Canny 边缘检测
    edges = cv2.Canny(
        (wall_mask * 255).astype(np.uint8),
        threshold1=50,
        threshold2=150
    )

    # Hough 变换
    lines = cv2.HoughLines(
        edges,
        rho=1,
        theta=np.deg2rad(angle_resolution),
        threshold=50
    )

    if lines is None:
        # 默认只处理水平和垂直方向
        return [0.0]

    # 统计各角度的直线数量
    angles = []
    for line in lines:
        theta = np.rad2deg(line[0][1])
        # 归一化到 0-180 度
        theta = theta % 180
        # 去除接近水平/垂直的（已经在默认处理）
        if theta > 5 and theta < 85:
            angles.append(theta)
        elif theta > 95 and theta < 175:
            angles.append(theta - 90)

    if not angles:
        return [0.0]

    # 找最常见的角度（用直方图）
    hist, bin_edges = np.histogram(angles, bins=180)
    top_bins = np.argsort(hist)[::-1][:n_angles]
    top_angles = [bin_edges[b] for b in top_bins if hist[b] > 0]

    # 始终包含 0 度（水平/垂直）
    if 0.0 not in top_angles:
        top_angles = [0.0] + top_angles

    return top_angles[:n_angles]


print('✓ Hough 角度检测函数定义完成')

## 5. Section 2.4 Step 3-4：提取单个墙体段

In [ ]:
def extract_wall_segments_at_angle(
        wall_mask: np.ndarray,
        angle_deg: float,
        min_area: int = 200
) -> List[np.ndarray]:
    """
    论文 Section 2.4 Step 3-4：
    1. 旋转 mask 使该角度的墙体变为水平/垂直
    2. 用形态学 opening 分离水平段和垂直段
    3. 用轮廓提取得到单个墙体段
    4. 验证算法：去除实际上是斜的墙体段
    返回原始坐标系下的轮廓列表
    """
    h, w = wall_mask.shape

    # 旋转 mask
    center = (w // 2, h // 2)
    M      = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    rotated = cv2.warpAffine(
        wall_mask.astype(np.uint8), M, (w, h),
        flags=cv2.INTER_NEAREST
    )

    segments_rotated = []

    # 分离水平段（长横向 kernel）
    kernel_h = cv2.getStructuringElement(
        cv2.MORPH_RECT, (20, 1)
    )
    horiz = cv2.morphologyEx(
        rotated, cv2.MORPH_OPEN, kernel_h
    )

    # 分离垂直段（长纵向 kernel）
    kernel_v = cv2.getStructuringElement(
        cv2.MORPH_RECT, (1, 20)
    )
    vert = cv2.morphologyEx(
        rotated, cv2.MORPH_OPEN, kernel_v
    )

    for component_mask in [horiz, vert]:
        contours, _ = cv2.findContours(
            component_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue

            # 论文：验证算法 - 用最小外接矩形检查是否真的是水平/垂直
            rect = cv2.minAreaRect(cnt)
            rect_angle = abs(rect[2])
            # 允许 10 度以内的倾斜
            if rect_angle > 10 and rect_angle < 80:
                continue  # 跳过真正斜的

            segments_rotated.append(cnt)

    # 反旋转：把轮廓坐标变回原始坐标系
    M_inv = cv2.getRotationMatrix2D(center, -angle_deg, 1.0)
    segments_original = []
    for cnt in segments_rotated:
        pts = cnt.reshape(-1, 2).astype(np.float32)
        pts_rot = cv2.transform(pts.reshape(1, -1, 2), M_inv)
        segments_original.append(
            pts_rot.reshape(-1, 1, 2).astype(np.int32)
        )

    return segments_original


print('✓ 墙体段提取函数定义完成')

## 6. Section 2.4 Step 5：Shrinking 算法

In [ ]:
def compute_iou_mask_box(mask: np.ndarray,
                          box: Tuple[int,int,int,int]) -> float:
    """计算 mask 和矩形 box 的 IoU"""
    x1, y1, x2, y2 = box
    h, w = mask.shape
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w, x2); y2 = min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return 0.0
    box_mask = np.zeros_like(mask)
    box_mask[y1:y2, x1:x2] = 1
    intersection = (mask & box_mask).sum()
    union        = (mask | box_mask).sum()
    return intersection / (union + 1e-8)


def shrinking_algorithm(
        wall_mask: np.ndarray,
        contour:   np.ndarray,
        iou_threshold: float = 0.85,
        min_size: int = 5
) -> Optional[Tuple[int,int,int,int]]:
    """
    论文 Section 2.4 Step 5：Shrinking 算法
    把不规则形状的墙体段转成最优矩形 bbox

    算法：
    1. 计算 axis-aligned bounding box
    2. 分别从四个方向各缩小 1 像素
    3. 选择 IoU 最大的方向作为新 bbox
    4. 重复直到 IoU 达到阈值或 bbox 太小
    """
    # 初始 bbox
    x, y, w_cnt, h_cnt = cv2.boundingRect(contour)
    box = (x, y, x + w_cnt, y + h_cnt)

    # 用轮廓创建局部 mask
    h, w = wall_mask.shape
    local_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(local_mask, [contour], -1, 1, -1)

    best_iou = compute_iou_mask_box(local_mask, box)

    improved = True
    while improved:
        improved = False
        x1, y1, x2, y2 = box

        if (x2 - x1) < min_size or (y2 - y1) < min_size:
            break

        # 四个方向各缩小 1 像素
        candidates = [
            (x1+1, y1,   x2,   y2),   # 左缩
            (x1,   y1+1, x2,   y2),   # 上缩
            (x1,   y1,   x2-1, y2),   # 右缩
            (x1,   y1,   x2,   y2-1), # 下缩
        ]

        for cand in candidates:
            if cand[2] - cand[0] < min_size:
                continue
            if cand[3] - cand[1] < min_size:
                continue
            iou = compute_iou_mask_box(local_mask, cand)
            if iou > best_iou:
                best_iou = iou
                box      = cand
                improved = True

        if best_iou >= iou_threshold:
            break

    # 论文：如果 bbox 之外还有大块墙体，对剩余部分递归处理
    # 简化版：直接返回当前 bbox
    x1, y1, x2, y2 = box
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return None
    return box


print('✓ Shrinking 算法定义完成')

## 7. Section 2.4 Step 6：重叠处理

In [ ]:
def box_area(box: Tuple) -> float:
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def box_intersection(b1: Tuple, b2: Tuple) -> float:
    ix1 = max(b1[0], b2[0]); iy1 = max(b1[1], b2[1])
    ix2 = min(b1[2], b2[2]); iy2 = min(b1[3], b2[3])
    return max(0, ix2-ix1) * max(0, iy2-iy1)


def resolve_overlapping_boxes(
        boxes: List[Tuple],
        wall_mask: np.ndarray,
        overlap_thresh: float = 0.1
) -> List[Tuple]:
    """
    论文 Section 2.4 Step 6：处理矢量化后重叠的 bbox

    规则：
    1. 小 box 完全被大 box 包含 → 删除小 box
    2. 部分重叠 → 缩小信息损失更小的那个 box
    """
    if not boxes:
        return []

    boxes = list(boxes)
    changed = True

    while changed:
        changed = False
        to_remove = set()
        n = len(boxes)

        for i in range(n):
            if i in to_remove:
                continue
            for j in range(i+1, n):
                if j in to_remove:
                    continue

                inter = box_intersection(boxes[i], boxes[j])
                if inter == 0:
                    continue

                area_i = box_area(boxes[i])
                area_j = box_area(boxes[j])

                # 情况 1：完全包含
                if inter >= area_i * 0.95:
                    # i 被 j 包含，删除 i
                    to_remove.add(i)
                    changed = True
                    break
                elif inter >= area_j * 0.95:
                    # j 被 i 包含，删除 j
                    to_remove.add(j)
                    changed = True
                    continue

                # 情况 2：部分重叠
                overlap_ratio = inter / min(area_i, area_j)
                if overlap_ratio < overlap_thresh:
                    continue

                # 缩小信息损失更小的那个
                # 信息损失 = 缩小后与原始 mask 的 IoU 下降
                # 简化：缩小面积更小的 box
                if area_i <= area_j:
                    # 缩小 i，去除重叠部分
                    xi1, yi1, xi2, yi2 = boxes[i]
                    xj1, yj1, xj2, yj2 = boxes[j]
                    # 沿重叠方向缩小
                    if xi2 > xj1 and xi1 < xj1:  # i 在 j 左边
                        boxes[i] = (xi1, yi1, xj1, yi2)
                    elif xi1 < xj2 and xi2 > xj2:  # i 在 j 右边
                        boxes[i] = (xj2, yi1, xi2, yi2)
                    elif yi2 > yj1 and yi1 < yj1:  # i 在 j 上边
                        boxes[i] = (xi1, yi1, xi2, yj1)
                    elif yi1 < yj2 and yi2 > yj2:  # i 在 j 下边
                        boxes[i] = (xi1, yj2, xi2, yi2)
                    changed = True
                else:
                    xi1, yi1, xi2, yi2 = boxes[i]
                    xj1, yj1, xj2, yj2 = boxes[j]
                    if xj2 > xi1 and xj1 < xi1:
                        boxes[j] = (xj1, yj1, xi1, yj2)
                    elif xj1 < xi2 and xj2 > xi2:
                        boxes[j] = (xi2, yj1, xj2, yj2)
                    elif yj2 > yi1 and yj1 < yi1:
                        boxes[j] = (xj1, yj1, xj2, yi1)
                    elif yj1 < yi2 and yj2 > yi2:
                        boxes[j] = (xj1, yi2, xj2, yj2)
                    changed = True

        boxes = [b for i, b in enumerate(boxes) if i not in to_remove]

    # 过滤无效 bbox
    boxes = [b for b in boxes if box_area(b) > 0]
    return boxes


print('✓ 重叠处理函数定义完成')

## 8. 完整矢量化流程（Section 2.4）

In [ ]:
def vectorize_wall_mask(
        wall_mask: np.ndarray,
        iou_threshold: float = 0.85,
        min_segment_area: int = 200,
        debug: bool = False
) -> List[Tuple[int,int,int,int]]:
    """
    论文 Section 2.4 完整矢量化流程
    输入：binary wall mask
    输出：矩形 bbox 列表 [(x1,y1,x2,y2), ...]
    """
    # Step 1: 形态学预处理
    mask = morphological_preprocessing(wall_mask)
    if debug:
        print(f'  预处理后 wall 像素: {mask.sum()}')

    # Step 2: 找墙体主要角度
    angles = find_wall_angles(mask)
    if debug:
        print(f'  检测到角度: {angles}')

    # Step 3-4: 按角度提取墙体段
    all_contours = []
    for angle in angles:
        contours = extract_wall_segments_at_angle(
            mask, angle, min_area=min_segment_area
        )
        all_contours.extend(contours)
    if debug:
        print(f'  提取到轮廓数: {len(all_contours)}')

    # Step 5: Shrinking 算法 → 每个轮廓变成矩形 bbox
    wall_boxes = []
    for cnt in all_contours:
        box = shrinking_algorithm(
            mask, cnt, iou_threshold=iou_threshold
        )
        if box is not None:
            wall_boxes.append(box)
    if debug:
        print(f'  Shrinking 后 bbox 数: {len(wall_boxes)}')

    # Step 6: 处理重叠
    wall_boxes = resolve_overlapping_boxes(wall_boxes, mask)
    if debug:
        print(f'  去重叠后 bbox 数: {len(wall_boxes)}')

    return wall_boxes


print('✓ 完整矢量化流程定义完成')

## 9. Section 2.5：3D 重建

In [ ]:
# 论文假设的尺寸（Section 2.5）
WALL_HEIGHT   = 2.8   # 米
DOOR_HEIGHT   = 2.1   # 米
WINDOW_HEIGHT = 1.2   # 米
WINDOW_SILL   = 0.9   # 窗台高度（距地面）
PIXELS_PER_METER = 50 # 像素/米（根据实际比例调整）


def match_openings_to_walls(
        wall_boxes:  List[Tuple],
        opening_boxes: np.ndarray,
        opening_labels: np.ndarray
) -> List[dict]:
    """
    论文 Section 2.5：
    把检测到的门窗匹配到对应的墙体段
    选择 overlap 最大的墙体
    """
    matched = []

    for i, (box, label) in enumerate(zip(opening_boxes, opening_labels)):
        x1, y1, x2, y2 = [int(v) for v in box]
        opening_area = (x2-x1) * (y2-y1)
        if opening_area <= 0:
            continue

        best_wall_idx = -1
        best_overlap  = 0

        for j, wall in enumerate(wall_boxes):
            inter = box_intersection(
                (x1, y1, x2, y2), wall
            )
            # 论文：选 overlap 最大的墙体
            if inter > best_overlap:
                best_overlap  = inter
                best_wall_idx = j

        if best_wall_idx >= 0:
            wall = wall_boxes[best_wall_idx]
            matched.append({
                'opening_box':   (x1, y1, x2, y2),
                'type':          'door' if label == 1 else 'window',
                'wall_idx':      best_wall_idx,
                'wall_box':      wall,
            })

    return matched


def build_3d_model(
        wall_boxes:   List[Tuple],
        openings:     List[dict],
        scale:        float = 1.0 / PIXELS_PER_METER,
        wall_height:  float = WALL_HEIGHT,
        door_height:  float = DOOR_HEIGHT,
        win_height:   float = WINDOW_HEIGHT,
        win_sill:     float = WINDOW_SILL,
        output_path:  str   = None
) -> Optional[object]:
    """
    论文 Section 2.5：3D 重建
    用矩形 box primitives 构建墙体，裁剪门窗开口
    输出 trimesh Scene，可导出 GLB/OBJ
    """
    if not HAS_TRIMESH:
        print('需要安装 trimesh: pip install trimesh')
        return None

    scene = trimesh.Scene()

    # 建立门窗索引（按墙体）
    wall_openings = {i: [] for i in range(len(wall_boxes))}
    for op in openings:
        wall_openings[op['wall_idx']].append(op)

    for wall_idx, wall in enumerate(wall_boxes):
        wx1, wy1, wx2, wy2 = wall

        # 像素 → 米
        x1_m = wx1 * scale
        y1_m = wy1 * scale
        x2_m = wx2 * scale
        y2_m = wy2 * scale
        width_m  = abs(x2_m - x1_m)
        depth_m  = abs(y2_m - y1_m)

        if width_m < 0.01 or depth_m < 0.01:
            continue

        # 判断墙体方向（水平还是垂直）
        is_horizontal = width_m > depth_m

        # 创建完整墙体 box
        wall_mesh = trimesh.creation.box(
            extents=[width_m, depth_m, wall_height]
        )
        # 移动到正确位置（z=0 在地面）
        cx = (x1_m + x2_m) / 2
        cy = (y1_m + y2_m) / 2
        wall_mesh.apply_translation([cx, cy, wall_height / 2])

        # 裁剪门窗开口
        for op in wall_openings[wall_idx]:
            ox1, oy1, ox2, oy2 = op['opening_box']
            ox1_m = ox1 * scale; ox2_m = ox2 * scale
            oy1_m = oy1 * scale; oy2_m = oy2 * scale

            op_w = abs(ox2_m - ox1_m)
            op_d = abs(oy2_m - oy1_m)

            if op['type'] == 'door':
                op_h     = door_height
                op_z_bot = 0.0
            else:  # window
                op_h     = win_height
                op_z_bot = win_sill

            # 创建开口 box（用于布尔减法）
            opening_box = trimesh.creation.box(
                extents=[op_w + 0.01, op_d + 0.01, op_h + 0.01]
            )
            ocx = (ox1_m + ox2_m) / 2
            ocy = (oy1_m + oy2_m) / 2
            oz  = op_z_bot + op_h / 2
            opening_box.apply_translation([ocx, ocy, oz])

            # 布尔减法：从墙体中减去开口
            try:
                wall_mesh = trimesh.boolean.difference(
                    [wall_mesh, opening_box],
                    engine='blender'
                )
            except Exception:
                # 布尔运算失败时跳过，保留完整墙体
                pass

        # 设置颜色
        wall_mesh.visual.face_colors = [200, 200, 200, 255]
        scene.add_geometry(
            wall_mesh, node_name=f'wall_{wall_idx}'
        )

    # 添加地板
    if wall_boxes:
        all_x = [b[0] for b in wall_boxes] + [b[2] for b in wall_boxes]
        all_y = [b[1] for b in wall_boxes] + [b[3] for b in wall_boxes]
        fx1 = min(all_x) * scale; fx2 = max(all_x) * scale
        fy1 = min(all_y) * scale; fy2 = max(all_y) * scale
        floor = trimesh.creation.box(
            extents=[fx2-fx1, fy2-fy1, 0.05]
        )
        floor.apply_translation([
            (fx1+fx2)/2, (fy1+fy2)/2, -0.025
        ])
        floor.visual.face_colors = [180, 160, 140, 255]
        scene.add_geometry(floor, node_name='floor')

    # 导出
    if output_path:
        ext = Path(output_path).suffix.lower()
        if ext == '.glb':
            scene.export(output_path)
        elif ext == '.obj':
            scene.export(output_path)
        print(f'✓ 3D 模型已导出: {output_path}')

    return scene


print('✓ 3D 重建函数定义完成')

## 10. 完整端到端后处理 Pipeline

In [ ]:
def postprocess_pipeline(
        image_path: str,
        output_dir: str,
        iou_threshold: float = 0.85,
        debug: bool = True
) -> dict:
    """
    完整端到端后处理流程
    输入：平面图路径
    输出：wall bbox 列表 + 3D 模型文件
    """
    os.makedirs(output_dir, exist_ok=True)
    name = Path(image_path).stem

    print(f'\n处理: {name}')
    t0 = time.time()

    # ── Step 1: 读取图片 ──
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f'图片不存在: {image_path}')
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    print(f'  图片尺寸: {img_rgb.shape[:2]}')

    # ── Step 2: 模型推理 ──
    pred = predict(img_rgb)
    wall_mask = pred['wall_mask']
    det_boxes  = pred['boxes']
    det_labels = pred['labels']
    print(f'  wall mask 覆盖率: {wall_mask.mean()*100:.1f}%')
    print(f'  检测到门窗: {len(det_boxes)} 个')

    # ── Step 3: 矢量化（Section 2.4）──
    print('  矢量化...')
    wall_boxes = vectorize_wall_mask(
        wall_mask, iou_threshold=iou_threshold, debug=debug
    )
    print(f'  矢量化得到 {len(wall_boxes)} 个墙体段')

    # ── Step 4: 门窗匹配 ──
    openings = match_openings_to_walls(
        wall_boxes, det_boxes, det_labels
    )
    doors   = [o for o in openings if o['type'] == 'door']
    windows = [o for o in openings if o['type'] == 'window']
    print(f'  匹配: {len(doors)} 门, {len(windows)} 窗')

    # ── Step 5: 可视化矢量化结果 ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img_rgb)
    axes[0].set_title('原始图片'); axes[0].axis('off')

    axes[1].imshow(wall_mask, cmap='gray')
    axes[1].set_title('预测 Wall Mask'); axes[1].axis('off')

    # 矢量化结果
    vis = img_rgb.copy()
    for box in wall_boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for box, label in zip(det_boxes, det_labels):
        x1, y1, x2, y2 = [int(v) for v in box]
        color = (0, 0, 255) if label == 1 else (255, 0, 0)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

    axes[2].imshow(vis)
    wall_p  = mpatches.Patch(color='green', label='Wall')
    door_p  = mpatches.Patch(color='blue',  label='Door')
    win_p   = mpatches.Patch(color='red',   label='Window')
    axes[2].legend(handles=[wall_p, door_p, win_p], loc='upper right')
    axes[2].set_title(
        f'矢量化结果: {len(wall_boxes)}墙 {len(doors)}门 {len(windows)}窗'
    )
    axes[2].axis('off')

    plt.tight_layout()
    vis_path = os.path.join(output_dir, f'{name}_vectorized.png')
    plt.savefig(vis_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  可视化已保存: {vis_path}')

    # ── Step 6: 3D 重建（Section 2.5）──
    model_3d = None
    if HAS_TRIMESH and wall_boxes:
        print('  3D 重建...')
        glb_path = os.path.join(output_dir, f'{name}_3d.glb')
        model_3d = build_3d_model(
            wall_boxes, openings,
            output_path=glb_path
        )

    # ── Step 7: 保存结果 JSON ──
    result = {
        'image':      image_path,
        'wall_boxes': [list(b) for b in wall_boxes],
        'doors':      [{'box': list(o['opening_box'])} for o in doors],
        'windows':    [{'box': list(o['opening_box'])} for o in windows],
        'stats': {
            'n_walls':   len(wall_boxes),
            'n_doors':   len(doors),
            'n_windows': len(windows),
            'elapsed':   round(time.time() - t0, 2),
        }
    }
    json_path = os.path.join(output_dir, f'{name}_result.json')
    with open(json_path, 'w') as f:
        json.dump(result, f, indent=2)

    print(f'  完成  耗时: {result["stats"]["elapsed"]}s')
    return result


print('✓ 完整 pipeline 定义完成')

## 11. 测试：对验证集样本运行完整流程

In [ ]:
# 测试用：用验证集里的图片
from numpy import genfromtxt

val_folders = genfromtxt(DATA_FOLDER + 'val.txt', dtype='str')
test_folder = val_folders[0].strip('/')
test_img_path = os.path.join(DATA_FOLDER, test_folder, 'F1_scaled.png')

print(f'测试图片: {test_img_path}')

result = postprocess_pipeline(
    image_path=test_img_path,
    output_dir=OUTPUT_DIR,
    iou_threshold=0.85,
    debug=True
)

print('\n结果摘要:')
print(f'  墙体段: {result["stats"]["n_walls"]}')
print(f'  门:     {result["stats"]["n_doors"]}')
print(f'  窗:     {result["stats"]["n_windows"]}')
print(f'  耗时:   {result["stats"]["elapsed"]}s')

## 12. 测试你公司的平面图

In [ ]:
# 替换成你公司平面图的路径
COMPANY_IMG_PATH = 'company_test/floorplan1.png'

if os.path.exists(COMPANY_IMG_PATH):
    result = postprocess_pipeline(
        image_path=COMPANY_IMG_PATH,
        output_dir=OUTPUT_DIR,
        iou_threshold=0.85,
        debug=True
    )
else:
    print(f'图片不存在: {COMPANY_IMG_PATH}')
    print('请修改 COMPANY_IMG_PATH 为你公司图片的实际路径')

## 13. 计算矢量化 IoU（论文 Table 1 的指标）

In [ ]:
def compute_vectorization_iou(
        wall_boxes: List[Tuple],
        gt_mask:    np.ndarray
) -> float:
    """
    论文 Table 1 的 IoU vect 指标：
    把矢量化 bbox 转成 binary mask，和 GT mask 计算 IoU
    论文结果：Ours IoU vect = 0.80
    """
    h, w = gt_mask.shape
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    for box in wall_boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        x1 = max(0, x1); y1 = max(0, y1)
        x2 = min(w, x2); y2 = min(h, y2)
        pred_mask[y1:y2, x1:x2] = 1

    intersection = (pred_mask & gt_mask).sum()
    union        = (pred_mask | gt_mask).sum()
    return intersection / (union + 1e-8)


# 批量评估验证集
sys.path.insert(0, CUBICASA_ROOT)
os.chdir(CUBICASA_ROOT)

val_folders = genfromtxt(DATA_FOLDER + 'val.txt', dtype='str')
N_EVAL = 20  # 评估前 20 个样本

mask_ious = []
vect_ious = []

print(f'评估前 {N_EVAL} 个验证样本...')
for folder in val_folders[:N_EVAL]:
    folder = folder.strip('/')
    img_path = os.path.join(DATA_FOLDER, folder, 'F1_scaled.png')
    svg_path = os.path.join(DATA_FOLDER, folder, 'model.svg')

    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    # GT mask
    try:
        house = House(svg_path, h, w)
        seg   = house.get_segmentation_tensor()
        gt_wall = (seg[0] == 2).astype(np.uint8)
    except:
        continue

    # 推理
    pred = predict(img_rgb)

    # IoU mask（直接分割结果）
    wall_pred = pred['wall_mask']
    inter = (wall_pred & gt_wall).sum()
    union = (wall_pred | gt_wall).sum()
    mask_iou = inter / (union + 1e-8)
    mask_ious.append(mask_iou)

    # IoU vect（矢量化后）
    wall_boxes = vectorize_wall_mask(wall_pred)
    vect_iou   = compute_vectorization_iou(wall_boxes, gt_wall)
    vect_ious.append(vect_iou)

print(f'\n评估结果（前 {N_EVAL} 个样本）:')
print(f'  IoU mask : {np.mean(mask_ious):.4f}  (论文: 0.81)')
print(f'  IoU vect : {np.mean(vect_ious):.4f}  (论文: 0.80)')
print(f'\n注：完整评估需要全部 400 个验证样本，将 N_EVAL 改为 400')

## 14. 后处理验证系统

```
验证 1：Wall Mask IoU     → 分割质量（目标 > 0.75）
验证 2：矢量化 IoU        → 矢量化质量（目标 > 0.60）
验证 3：检测 P/R/F1       → 门窗检测质量（目标 F1 > 0.70）
验证 4：门窗墙体匹配率    → 语义合理性
验证 5：3D 几何合理性     → bbox 宽高比合理
验证 6：批量评估 400 样本 → 统计汇总表
验证 7：失败案例分析      → 最差样本可视化
```

In [ ]:
from tqdm import tqdm
from torchvision.ops import box_iou as tv_box_iou

# ══════════════════════════════════════════════════════════════
# 验证工具函数
# ══════════════════════════════════════════════════════════════

WALL_CLASS_ID = 2
CROP_PADDING  = 10
MIN_CROP_SIZE = 64


def _crop_to_wall(image_rgb, gt_wall):
    """论文预处理：用GT wall mask裁剪图片和mask，保持评估与训练一致"""
    h, w = image_rgb.shape[:2]
    if gt_wall.sum() == 0:
        return None, None
    rows = np.where(gt_wall.any(axis=1))[0]
    cols = np.where(gt_wall.any(axis=0))[0]
    y1 = max(rows[0]  - CROP_PADDING, 0)
    y2 = min(rows[-1] + CROP_PADDING + 1, h)
    x1 = max(cols[0]  - CROP_PADDING, 0)
    x2 = min(cols[-1] + CROP_PADDING + 1, w)
    if (y2-y1) < MIN_CROP_SIZE or (x2-x1) < MIN_CROP_SIZE:
        return None, None
    return image_rgb[y1:y2, x1:x2], gt_wall[y1:y2, x1:x2]


def _iou_mask(pred, gt):
    """像素级 IoU"""
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return float(inter) / (float(union) + 1e-8)


def _vect_mask(wall_boxes, shape):
    """把 wall bbox 列表转成 binary mask"""
    h, w = shape
    m = np.zeros((h, w), dtype=np.uint8)
    for b in wall_boxes:
        x1,y1,x2,y2 = [int(v) for v in b]
        x1=max(0,x1); y1=max(0,y1)
        x2=min(w,x2); y2=min(h,y2)
        m[y1:y2, x1:x2] = 1
    return m


def _det_metrics(pred_boxes, pred_scores, pred_labels,
                  gt_boxes, gt_labels, iou_thresh=0.5, score_thresh=0.5):
    """单张图片的检测指标：TP/FP/FN"""
    keep = pred_scores >= score_thresh
    pb   = pred_boxes[keep];  pl = pred_labels[keep]
    tp = fp = fn = 0
    if len(gt_boxes) == 0:
        return len(pb), 0, 0
    if len(pb) == 0:
        return 0, 0, len(gt_boxes)
    iou_mat = tv_box_iou(
        torch.tensor(pb, dtype=torch.float32),
        torch.tensor(gt_boxes, dtype=torch.float32)
    ).numpy()
    matched = set()
    for i in range(len(pb)):
        if iou_mat.shape[1] == 0:
            fp += 1; continue
        j = iou_mat[i].argmax()
        if iou_mat[i, j] >= iou_thresh and j not in matched:
            tp += 1; matched.add(j)
        else:
            fp += 1
    fn = len(gt_boxes) - len(matched)
    return tp, fp, fn


def _match_rate(det_boxes, det_labels, wall_mask,
                 score_thresh=0.5, overlap_thresh=0.3):
    """门窗落在墙体上的比例"""
    h, w = wall_mask.shape
    door_m = door_t = win_m = win_t = 0
    for box, label in zip(det_boxes, det_labels):
        x1=max(0,int(box[0])); y1=max(0,int(box[1]))
        x2=min(w-1,int(box[2])); y2=min(h-1,int(box[3]))
        if x2<=x1 or y2<=y1: continue
        area    = (x2-x1)*(y2-y1)
        overlap = wall_mask[y1:y2, x1:x2].sum() / (area + 1e-8)
        matched = overlap >= overlap_thresh
        if label == 1:   door_t += 1; door_m += int(matched)
        elif label == 2: win_t  += 1; win_m  += int(matched)
    return door_m, door_t, win_m, win_t


# ══════════════════════════════════════════════════════════════
# 验证 1-5：单样本可视化验证（快速检查）
# ══════════════════════════════════════════════════════════════

def validate_single_sample(folder: str, debug: bool = True) -> dict:
    """
    对单个样本做完整后处理并逐阶段验证
    包含：mask IoU / vect IoU / 检测 P/R/F1 / 匹配率 / 3D 合理性
    """
    folder   = folder.strip('/')
    img_path = os.path.join(DATA_FOLDER, folder, 'F1_scaled.png')
    svg_path = os.path.join(DATA_FOLDER, folder, 'model.svg')

    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return {'folder': folder, 'status': 'no_image'}
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w    = img_rgb.shape[:2]

    house   = House(svg_path, h, w)
    seg     = house.get_segmentation_tensor()
    gt_wall = (seg[0] == WALL_CLASS_ID).astype(np.uint8)

    # GT 门窗 bbox
    gt_boxes_list, gt_labels_list = [], []
    for cls_id, lbl in [(2,1),(1,2)]:
        m = (seg[1] == cls_id).astype(np.uint8)
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in cnts:
            if cv2.contourArea(cnt) < 100: continue
            x,y,bw,bh = cv2.boundingRect(cnt)
            gt_boxes_list.append([x,y,x+bw,y+bh])
            gt_labels_list.append(lbl)
    gt_boxes  = np.array(gt_boxes_list,  dtype=np.float32) if gt_boxes_list else np.zeros((0,4))
    gt_labels = np.array(gt_labels_list, dtype=np.int64)   if gt_labels_list else np.zeros(0, dtype=np.int64)

    # 论文预处理：用 GT 裁剪
    img_crop, gt_crop = _crop_to_wall(img_rgb, gt_wall)
    if img_crop is None:
        return {'folder': folder, 'status': 'no_wall'}

    # 推理
    pred      = predict(img_crop)
    wall_pred = pred['wall_mask']
    if wall_pred.shape != gt_crop.shape:
        wall_pred = cv2.resize(
            wall_pred.astype(np.float32),
            (gt_crop.shape[1], gt_crop.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(np.uint8)

    # ── 验证 1：Wall Mask IoU ──
    iou_mask = _iou_mask(wall_pred, gt_crop)

    # ── 验证 2：矢量化 IoU ──
    wall_boxes = vectorize_wall_mask(wall_pred, iou_threshold=0.85)
    vect_m     = _vect_mask(wall_boxes, gt_crop.shape)
    iou_vect   = _iou_mask(vect_m, gt_crop)
    vect_pixel_ratio = vect_m.sum() / (wall_pred.sum() + 1e-8)

    # ── 验证 3：门窗检测 ──
    det_boxes  = pred['boxes']
    det_scores = pred['scores']
    det_labels = pred['labels']
    tp, fp, fn = _det_metrics(
        det_boxes, det_scores, det_labels, gt_boxes, gt_labels
    )
    prec = tp/(tp+fp+1e-8); rec = tp/(tp+fn+1e-8)
    f1   = 2*prec*rec/(prec+rec+1e-8)

    # ── 验证 4：门窗墙体匹配率 ──
    door_m,door_t,win_m,win_t = _match_rate(det_boxes, det_labels, wall_pred)
    door_match = door_m/(door_t+1e-8)
    win_match  = win_m/(win_t+1e-8)

    # ── 验证 5：3D 重建合理性 ──
    openings = match_openings_to_walls(wall_boxes, det_boxes, det_labels)
    doors_3d   = [o for o in openings if o['type']=='door']
    windows_3d = [o for o in openings if o['type']=='window']
    # 几何合理性：bbox 宽高比应在 0.1-10 之间
    aspect_ok = 0
    for b in wall_boxes:
        wr = (b[2]-b[0])/(b[3]-b[1]+1e-8)
        if 0.1 < wr < 10: aspect_ok += 1
    geom_ok_rate = aspect_ok / (len(wall_boxes)+1e-8)

    result = {
        'folder':         folder,
        'status':         'ok',
        'iou_mask':       iou_mask,
        'iou_vect':       iou_vect,
        'n_wall_boxes':   len(wall_boxes),
        'vect_pixel_ratio': vect_pixel_ratio,
        'det_precision':  prec,
        'det_recall':     rec,
        'det_f1':         f1,
        'door_match':     door_match,
        'win_match':      win_match,
        'n_doors_3d':     len(doors_3d),
        'n_windows_3d':   len(windows_3d),
        'geom_ok_rate':   geom_ok_rate,
        # 保存中间结果用于可视化
        '_img_crop':  img_crop,
        '_gt_crop':   gt_crop,
        '_wall_pred': wall_pred,
        '_vect_m':    vect_m,
        '_wall_boxes': wall_boxes,
        '_det_boxes':  det_boxes,
        '_det_labels': det_labels,
    }

    if debug:
        print(f'\n{'='*50}')
        print(f'样本: {folder}')
        print(f'{'='*50}')
        s1 = '✓' if iou_mask > 0.75 else '✗'
        s2 = '✓' if iou_vect > 0.60 else '✗'
        s3 = '✓' if f1 > 0.70 else '✗'
        s4 = '✓' if (door_match+win_match)/2 > 0.70 else '✗'
        s5 = '✓' if geom_ok_rate > 0.90 else '✗'
        print(f'  {s1} 验证1 IoU mask      : {iou_mask:.4f}  (目标 > 0.75)')
        print(f'  {s2} 验证2 IoU vect      : {iou_vect:.4f}  (目标 > 0.60)')
        print(f'       wall boxes       : {len(wall_boxes)}  pixel ratio={vect_pixel_ratio:.2f}')
        print(f'  {s3} 验证3 检测 F1       : {f1:.4f}  P={prec:.3f} R={rec:.3f}')
        print(f'  {s4} 验证4 门匹配率      : {door_match:.4f}  ({door_m}/{door_t})')
        print(f'       窗匹配率      : {win_match:.4f}  ({win_m}/{win_t})')
        print(f'  {s5} 验证5 3D几何合理性  : {geom_ok_rate:.4f}  '
              f'({len(doors_3d)}门 {len(windows_3d)}窗)')

    return result


def visualize_validation(result: dict, save_path: str = None):
    """可视化单个样本的验证结果（6宫格）"""
    if result.get('status') != 'ok':
        print(f'样本状态异常: {result.get("status")}')
        return

    img   = result['_img_crop']
    gt    = result['_gt_crop']
    pred  = result['_wall_pred']
    vect  = result['_vect_m']
    wboxes = result['_wall_boxes']
    dboxes = result['_det_boxes']
    dlabels = result['_det_labels']

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # 原图
    axes[0,0].imshow(img)
    axes[0,0].set_title('原始图片（裁剪后）'); axes[0,0].axis('off')

    # GT wall mask
    axes[0,1].imshow(gt, cmap='gray')
    axes[0,1].set_title(f'GT Wall Mask\n({gt.sum()} px)')
    axes[0,1].axis('off')

    # 预测 wall mask + IoU
    axes[0,2].imshow(pred, cmap='gray')
    axes[0,2].set_title(
        f'预测 Wall Mask\nIoU mask={result["iou_mask"]:.4f}'
    )
    axes[0,2].axis('off')

    # 矢量化结果
    axes[1,0].imshow(vect, cmap='gray')
    axes[1,0].set_title(
        f'矢量化 Mask\nIoU vect={result["iou_vect"]:.4f}  '
        f'{result["n_wall_boxes"]}boxes'
    )
    axes[1,0].axis('off')

    # 差异图（pred vs GT）
    diff = np.zeros((*gt.shape, 3), dtype=np.uint8)
    diff[(pred==1)&(gt==1)]  = [255,255,0]   # 黄=TP
    diff[(pred==1)&(gt==0)]  = [255,0,0]     # 红=FP
    diff[(pred==0)&(gt==1)]  = [0,0,255]     # 蓝=FN
    axes[1,1].imshow(diff)
    tp_px = int(((pred==1)&(gt==1)).sum())
    fp_px = int(((pred==1)&(gt==0)).sum())
    fn_px = int(((pred==0)&(gt==1)).sum())
    axes[1,1].set_title(
        f'像素差异图\n黄=TP({tp_px}) 红=FP({fp_px}) 蓝=FN({fn_px})'
    )
    axes[1,1].axis('off')

    # 门窗检测结果
    vis = img.copy()
    for b in wboxes:
        cv2.rectangle(vis,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,255,0),2)
    for b,l in zip(dboxes, dlabels):
        c = (0,0,255) if l==1 else (255,0,0)
        cv2.rectangle(vis,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),c,2)
    axes[1,2].imshow(vis)
    wp = mpatches.Patch(color='green', label='Wall bbox')
    dp = mpatches.Patch(color='blue',  label='Door')
    wp2= mpatches.Patch(color='red',   label='Window')
    axes[1,2].legend(handles=[wp,dp,wp2], loc='upper right', fontsize=8)
    axes[1,2].set_title(
        f'检测结果\nF1={result["det_f1"]:.3f}  '
        f'door_match={result["door_match"]:.3f}'
    )
    axes[1,2].axis('off')

    plt.suptitle(
        f'后处理验证  {result["folder"]}', fontsize=13
    )
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f'  验证图已保存: {save_path}')
    plt.show()


print('✓ 验证工具函数定义完成')


## 15. 快速验证（单样本）

In [ ]:
# ══════════════════════════════════════════════════════════════
# 快速验证：对单个样本运行完整验证并可视化
# 建议在批量评估之前先运行这个确认流程正常
# ══════════════════════════════════════════════════════════════

val_folders = genfromtxt(DATA_FOLDER + 'val.txt', dtype='str')

# 取前 3 个样本逐一验证
for i in range(3):
    folder = val_folders[i]
    result = validate_single_sample(folder, debug=True)
    if result.get('status') == 'ok':
        vis_path = os.path.join(
            OUTPUT_DIR,
            f'validation_sample_{i}.png'
        )
        visualize_validation(result, save_path=vis_path)


## 16. 批量评估（400 样本）

In [ ]:
# ══════════════════════════════════════════════════════════════
# 验证 6：批量评估（400 个验证样本）
# ══════════════════════════════════════════════════════════════

def batch_validate(
        data_folder: str,
        n_samples:   int  = 400,
        save_dir:    str  = None,
) -> dict:
    """
    批量评估后处理质量
    返回各指标的均值/标准差/分位数
    """
    val_folders = genfromtxt(data_folder + 'val.txt', dtype='str')
    folders     = val_folders[:n_samples]

    records = []
    skip    = 0

    for folder in tqdm(folders, desc=f'批量验证({n_samples}样本)'):
        r = validate_single_sample(folder, debug=False)
        if r.get('status') != 'ok':
            skip += 1
            continue
        # 只保留数值字段
        records.append({
            k: v for k, v in r.items()
            if not k.startswith('_') and isinstance(v, (int, float))
        })

    n = len(records)
    print(f'\n有效样本: {n}/{len(folders)}  跳过: {skip}')

    # 计算统计量
    metrics = [
        'iou_mask', 'iou_vect', 'n_wall_boxes',
        'det_precision', 'det_recall', 'det_f1',
        'door_match', 'win_match', 'geom_ok_rate',
    ]
    summary = {}
    for m in metrics:
        vals = [r[m] for r in records if m in r]
        if not vals: continue
        a = np.array(vals)
        summary[m] = {
            'mean':  float(a.mean()),
            'std':   float(a.std()),
            'p25':   float(np.percentile(a, 25)),
            'p50':   float(np.percentile(a, 50)),
            'p75':   float(np.percentile(a, 75)),
            'p10':   float(np.percentile(a, 10)),
        }

    # 打印汇总表
    print(f'\n{"="*65}')
    print(f'批量评估结果（n={n}）')
    print(f'{"-"*65}')
    print(f'{"指标":<20} {"均值":>8} {"std":>7} {"p25":>7} {"p50":>7} {"p75":>7}')
    print(f'{"-"*65}')
    targets = {
        'iou_mask':    0.81,
        'iou_vect':    0.80,
        'det_f1':      0.70,
        'geom_ok_rate':0.90,
    }
    for m, s in summary.items():
        target = targets.get(m)
        flag   = ''
        if target is not None:
            flag = ' ✓' if s['mean'] >= target else ' ✗'
        print(
            f'{m:<20} {s["mean"]:>8.4f} {s["std"]:>7.4f} '
            f'{s["p25"]:>7.4f} {s["p50"]:>7.4f} {s["p75"]:>7.4f}{flag}'
        )
    print(f'{"-"*65}')
    print(f'论文目标: IoU mask=0.81  IoU vect=0.80')
    print(f'{"-"*65}')

    # 保存 JSON
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        path = os.path.join(save_dir, 'batch_validation.json')
        with open(path, 'w') as f:
            json.dump({'summary': summary, 'n_valid': n, 'n_skip': skip}, f, indent=2)
        print(f'  结果已保存: {path}')

    return {'summary': summary, 'records': records}


# ── 运行批量评估 ──
N_EVAL = 400   # 改为更小的数快速测试，例如 20

batch_result = batch_validate(
    data_folder = DATA_FOLDER,
    n_samples   = N_EVAL,
    save_dir    = OUTPUT_DIR,
)


## 17. 失败案例分析

In [ ]:
# ══════════════════════════════════════════════════════════════
# 验证 7：失败案例分析
# 找出 IoU 最低的样本，可视化对比，帮助定位问题
# ══════════════════════════════════════════════════════════════

def analyze_failures(
        records:    list,
        n_worst:    int  = 5,
        metric:     str  = 'iou_mask',
        output_dir: str  = None,
) -> None:
    """
    找出指定指标最差的 n 个样本并可视化
    metric: 'iou_mask' | 'iou_vect' | 'det_f1' | 'door_match'
    """
    if not records:
        print('无记录可分析')
        return

    # 按指标升序排列（最差在前）
    sorted_r = sorted(
        [r for r in records if metric in r],
        key=lambda x: x[metric]
    )
    worst = sorted_r[:n_worst]

    print(f'\n{"="*55}')
    print(f'最差 {n_worst} 个样本（按 {metric} 升序）')
    print(f'{"-"*55}')
    print(f'{"folder":<28} {metric:>10} {"iou_mask":>10} {"iou_vect":>10}')
    print(f'{"-"*55}')
    for r in worst:
        print(
            f'{r["folder"]:<28} '
            f'{r[metric]:>10.4f} '
            f'{r.get("iou_mask",0):>10.4f} '
            f'{r.get("iou_vect",0):>10.4f}'
        )

    # 可视化最差的 3 个样本
    n_vis = min(3, len(worst))
    print(f'\n可视化最差 {n_vis} 个样本...')
    for i, r in enumerate(worst[:n_vis]):
        # 重新运行验证获取中间结果
        full_r = validate_single_sample(r['folder'], debug=False)
        if full_r.get('status') != 'ok':
            continue
        save_path = None
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            save_path = os.path.join(
                output_dir,
                f'failure_{i+1}_{r["folder"].replace("/","_")}.png'
            )
        visualize_validation(full_r, save_path=save_path)


# ── 运行失败案例分析 ──
if 'batch_result' in dir() and batch_result['records']:
    analyze_failures(
        records    = batch_result['records'],
        n_worst    = 5,
        metric     = 'iou_mask',
        output_dir = OUTPUT_DIR,
    )
else:
    print('请先运行批量评估（上方 Cell）')
